# The Financial Advisory Agent

**Workshop:** 10 AI Agents Every AI Engineer Must Build — Agent 6
**Book:** *30 Agents Every AI Engineer Must Build* — Imran Ahmad (Packt Publishing, 2026)
**Reference:** Chapter 14, Section 14.1 (pp. 392–408)

---

In this notebook you will build **RetailAdvisor**, a supervised multi-agent system for retail financial advisory. A central Supervisor Agent routes client queries through a LangGraph `StateGraph` to specialist agents — Market Data, Financial Analysis, and News — while a composite risk framework scores positions on annualized volatility, maximum drawdown, and Value at Risk. A personalized planning layer then converts a client risk profile into a tailored portfolio recommendation, and every recommendation must pass a structural compliance gate before it can reach the client. Market-data analysis, risk assessment, and personalized planning are combined into one adviser: a risk profile in, a tailored portfolio recommendation out.

The notebook runs in **Simulation Mode** by default — no API keys required — using chapter-faithful mock data. Copy `../.env.template` to `.env` and add keys to switch individual services to live mode.

> ⚠️ **Disclaimer:** This agent is an educational demonstration. Financial outputs are illustrative and must not be treated as investment advice.


In [ ]:
# Google Colab bootstrap — runs only on Colab, no-op everywhere else.
# Locally you are already inside the agent folder with requirements installed.
import os
import sys

if "google.colab" in sys.modules:
    AGENT_DIR = "06-financial-advisory-agent"
    if not os.path.exists("/content/repo"):
        os.system("git clone --depth 1 https://github.com/cloudanum/ws-10-agents /content/repo")
    os.chdir(f"/content/repo/{AGENT_DIR}")
    # On Colab, prefer requirements-colab.txt when present: it drops pins that
    # cannot coexist with Colab's preinstalled stack (e.g. langchain 0.2.16
    # requires numpy<2 on Python 3.13, while Colab ships numpy 2.x).
    req_file = "requirements-colab.txt" if os.path.exists("requirements-colab.txt") else "requirements.txt"
    # Keep Colab's preinstalled scientific/kernel stack: the kernel already has
    # numpy, pandas, pydantic and ipykernel loaded, so letting pip replace them
    # (e.g. building numpy 1.26.4 from source or upgrading ipykernel) breaks the
    # running kernel with ABI errors or an OOM kill. Filter those lines out of
    # requirements and constrain the rest of the install to the installed versions.
    import re
    from importlib.metadata import PackageNotFoundError, version
    filtered = [
        line for line in open(req_file)
        if not re.match(r"\s*(numpy|pandas|pydantic|jupyter|ipykernel)\b", line, re.IGNORECASE)
    ]
    with open("/tmp/colab_requirements.txt", "w") as fh:
        fh.writelines(filtered)
    pins = []
    for pkg in ("numpy", "pandas", "pydantic", "ipykernel"):
        try:
            pins.append(f"{pkg}=={version(pkg)}")
        except PackageNotFoundError:
            pass
    with open("/tmp/colab_constraints.txt", "w") as fh:
        fh.write("\n".join(pins) + "\n")
    import subprocess
    res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "-r", "/tmp/colab_requirements.txt",
         "--constraint", "/tmp/colab_constraints.txt"],
        capture_output=True, text=True,
    )
    if res.returncode != 0:
        print("pip install failed — re-running without -q for the full resolver report:\n")
        subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "-r", "/tmp/colab_requirements.txt",
             "--constraint", "/tmp/colab_constraints.txt"],
        )
        raise RuntimeError("Colab bootstrap: pip install failed (see resolver report above)")
    print(f"Colab setup complete ({req_file}) — working directory: {os.getcwd()}")
else:
    print("Not on Colab — skipping bootstrap (local setup already in place).")

## Cell 0: Setup and Configuration

**Ref:** Technical Requirements (p.392)

This cell loads environment variables, detects API key availability per service, and configures the notebook to run in either **Live Mode** (with real APIs) or **Simulation Mode** (with chapter-faithful mock data).

In [1]:
# Cell 0: Setup and Configuration
# Ref: Technical Requirements (p.392)
# Author: Imran Ahmad

import os
import sys
import json
import operator
import warnings
from functools import partial
from typing import Annotated, Sequence, TypedDict, Literal, List

import numpy as np
import pandas as pd

from dotenv import load_dotenv
load_dotenv()

from mock_llm import (
    ColorLogger,
    ServiceConfig,
    graceful_fallback,
    MockChatOpenAI,
    MockStructuredChain,
    MockEmbeddingModel,
    MockVectorStore,
)

from mock_data import (
    MOCK_STOCK_DATA,
    MOCK_FINNHUB_QUOTES,
    MOCK_FINNHUB_FINANCIALS,
    generate_mock_price_history,
    MOCK_TAVILY_NEWS,
    MOCK_CLIENT_PROFILES,
    MOCK_LEGAL_CASES,
    MOCK_CONTRACT,
    MOCK_INTER_AGENT_MESSAGE,
)

warnings.filterwarnings("ignore", category=DeprecationWarning)

config = ServiceConfig()
logger = ColorLogger("Chapter14")

# Conditional LLM selection — Ref: Technical Requirements (p.392)
if config.is_live("OPENAI_API_KEY"):
    try:
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(model="gpt-4o-mini-2024-07-18", temperature=0)
        logger.success("Using LIVE OpenAI LLM (gpt-4o-mini-2024-07-18)")
    except Exception as e:
        logger.error(f"ChatOpenAI init failed: {e}. Falling back to MockChatOpenAI.")
        llm = MockChatOpenAI(model="gpt-4o-mini-2024-07-18", temperature=0)
else:
    llm = MockChatOpenAI(model="gpt-4o-mini-2024-07-18", temperature=0)
    logger.info("Using SIMULATED LLM (MockChatOpenAI)")

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_core.tools import tool
from pydantic import BaseModel
from langgraph.prebuilt import create_react_agent
from langgraph.graph import END, START, StateGraph

# Version compatibility: create_react_agent's system-instructions argument is
# "state_modifier" on langgraph 0.2.x (book-pinned) and "prompt" on langgraph
# 1.x (preinstalled on Colab). Use whichever the installed langgraph accepts.
import inspect as _inspect

def make_react_agent(llm, tools, instructions):
    params = _inspect.signature(create_react_agent).parameters
    kw = "state_modifier" if "state_modifier" in params else "prompt"
    return create_react_agent(llm, tools=tools, **{kw: instructions})


logger.success("Setup complete — all imports loaded")


══════════════════════════════════════════════════════
  CHAPTER 14 — SERVICE STATUS DASHBOARD
  Book: 30 Agents Every AI Engineer Must Build
  Author: Imran Ahmad
══════════════════════════════════════════════════════
  OpenAI (LLM)                        ○ SIMULATED
  Anthropic (LLM)                     ○ SIMULATED
  Google Gemini (LLM)                 ○ SIMULATED
  Ollama Local (LLM)                  ○ SIMULATED
  Finnhub (Financial Data)            ○ SIMULATED
  Tavily (News Search)                ○ SIMULATED
══════════════════════════════════════════════════════

[18:22:44] [Chapter14] INFO [Simulation Mode] MockChatOpenAI initialized (model=gpt-4o-mini-2024-07-18)
[18:22:44] [Chapter14] INFO Using SIMULATED LLM (MockChatOpenAI)
[18:22:44] [Chapter14] SUCCESS Setup complete — all imports loaded


/Users/iahmad/Creator/Courses_and_conferences/Packt/10_agents/10-agents-workshop/06-financial-advisory-agent/.venv/lib/python3.14/site-packages/langgraph/checkpoint/base/__init__.py:24: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
# Multi-provider LLM support (OpenAI / Anthropic / Google Gemini)
# Set LLM_PROVIDER in .env to choose: openai | anthropic | google | auto
# Auto-detection uses the first available key.
# See supporting/llm_provider.py for details.

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, '..')

try:
    from supporting.llm_provider import detect_provider, get_llm, PROVIDER_MODELS, print_provider_banner
    _PROVIDER, _PROVIDER_KEY, _PROVIDER_MODE = detect_provider()
    print_provider_banner(_PROVIDER, _PROVIDER_MODE)
except ImportError:
    print('[INFO] supporting/llm_provider.py not found — using default OpenAI path')
    _PROVIDER, _PROVIDER_KEY, _PROVIDER_MODE = 'openai', os.getenv('OPENAI_API_KEY'), 'LIVE' if os.getenv('OPENAI_API_KEY') else 'SIMULATION'



   SIMULATION MODE ACTIVE
   Using MockLLM — no API key required



## Cell 1: Supervisor Architecture

**Ref:** Section 14.1, Figure 14.1 (pp. 393–395)

The Financial Advisory Agent uses a **supervised multi-agent architecture** (Figure 14.1). A central Supervisor Agent serves as both entry point and policy-aware orchestrator, deciding which specialist to invoke, in what order, and with what state recorded at each step.

**Specialist agents:** Market Data Agent, Financial Analysis Agent, News Agent

**Figure 14.1 — Multi-agent architecture for the Financial Advisory Agent:**

```
                  ┌─────────────────┐
                  │ Analysis Agent  │
                  │(Financial       │
                  │ Metrics)        │
                  └───────┬─────────┘
                          │
                          ▼
┌─────────────────┐ ┌───────────┐ ┌─────────────────┐
│ Market Data     │←│Supervisor │→│   Risk Agent    │
│ Agent           │ │   Agent   │ │  (VaR /         │
│(yfinance /      │ │           │ │   Volatility)   │
│ Finnhub)        │ └─────┬─────┘ └─────────────────┘
└─────────────────┘       │
                    FINISH (Post-Audit)
                          │
                          ▼
                  ┌───────────────┐
                  │  Compliance   │
                  │  Validated    │
                  │Recommendation│
                  └───────────────┘
```

The state-graph routing mechanism is the key architectural safeguard — it turns the advisory process into a traceable sequence of states and transitions, making it possible to enforce tool permissions, require human checkpoints, and attach an audit trail to every recommendation.

In [3]:
# Cell 1: Supervisor Architecture
# Ref: Section 14.1, Figure 14.1 (p.393-395)
# Author: Imran Ahmad

class RouteResponse(BaseModel):
    next: Literal[
        "Market_Data_Agent",
        "Financial_Analysis_Agent",
        "News_Agent",
        "FINISH"
    ]

members = ["Market_Data_Agent", "Financial_Analysis_Agent", "News_Agent"]

system_prompt = (
    "You are a Financial Services Supervisor managing: "
    f"{', '.join(members)}. "
    "Route queries to the appropriate specialist. "
    "Use Market_Data_Agent for price and volume data. "
    "Use Financial_Analysis_Agent for financial computations. "
    "Use News_Agent for market news and sentiment. "
    "Select FINISH when the query is fully resolved."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="messages"),
    ("system", "Choose the next agent from: {options}.")
]).partial(options=str(members + ["FINISH"]))

def supervisor_agent(state):
    """Route to the next specialist agent via structured output.
    Ref: Section 14.1, p.395"""
    chain = prompt | llm.with_structured_output(RouteResponse)
    return {"next": chain.invoke(state).next}

logger.success("Supervisor architecture initialized")
logger.info(f"Agent team: {members}")

[18:22:44] [Chapter14] SUCCESS Supervisor architecture initialized
[18:22:44] [Chapter14] INFO Agent team: ['Market_Data_Agent', 'Financial_Analysis_Agent', 'News_Agent']


## Cell 2: Market Data Agent

**Ref:** Section 14.1.1 (p.395–396)

The Market Data Agent wraps the `yfinance` library to retrieve real-time stock information. The `@graceful_fallback` decorator ensures that if the live API fails, the agent falls back to chapter-derived mock data.

In [4]:
# Cell 2: Market Data Agent
# Ref: Section 14.1.1, p.395-396
# Author: Imran Ahmad

@tool
def get_market_data(query: str) -> str:
    """Retrieve current market data for a given stock symbol.
    Ref: Section 14.1.1, p.395"""
    stock_symbol = query.strip().upper()

    if config.is_live("OPENAI_API_KEY"):
        try:
            import yfinance as yf
            ticker = yf.Ticker(stock_symbol)
            info = ticker.info
            if not info or not info.get("currentPrice"):
                raise ValueError("Empty response")
            logger.success(f"[Market Data] LIVE data for {stock_symbol}")
        except Exception:
            info = MOCK_STOCK_DATA.get(stock_symbol, MOCK_STOCK_DATA["AAPL"])
            logger.info(f"[Market Data] Fallback to mock for {stock_symbol}")
    else:
        info = MOCK_STOCK_DATA.get(stock_symbol, MOCK_STOCK_DATA["AAPL"])
        logger.info(f"[Market Data] SIMULATED data for {stock_symbol}")

    return (
        f"Market Data for {stock_symbol}: "
        f"Price: ${info.get('currentPrice', 'N/A')}, "
        f"Market Cap: ${info.get('marketCap', 'N/A')}, "
        f"P/E Ratio: {info.get('trailingPE', 'N/A')}, "
        f"Day Range: ${info.get('dayLow', 'N/A')}-${info.get('dayHigh', 'N/A')}, "
        f"Volume: {info.get('volume', 'N/A')}"
    )

market_agent = make_react_agent(
    llm, [get_market_data],
    "You are the Market Data Agent. "
    "Retrieve real-time stock data for client queries."
)

# Demo
print(get_market_data.invoke("AAPL"))
logger.success("Market Data Agent initialized")

[18:22:44] [Chapter14] INFO [Simulation Mode] MockChatOpenAI initialized (model=gpt-4o-mini-2024-07-18)
[18:22:44] [Chapter14] INFO [Market Data] SIMULATED data for AAPL
Market Data for AAPL: Price: $178.72, Market Cap: $2800000000000, P/E Ratio: 28.5, Day Range: $176.5-$179.8, Volume: 52340000
[18:22:44] [Chapter14] SUCCESS Market Data Agent initialized


## Cell 3: Finnhub Integration — Portfolio Analysis

**Ref:** Section 14.1.1 (p.396)

For production deployments, the Finnhub API provides endpoints for basic financials, company metrics, real-time quotes, and company news.

In [5]:
# Cell 3: Finnhub Integration — Portfolio Analysis
# Ref: Section 14.1.1, p.396
# Author: Imran Ahmad

finnhub_client = None
if config.is_live("FINNHUB_API_KEY"):
    try:
        import finnhub
        finnhub_client = finnhub.Client(api_key=config.get_key("FINNHUB_API_KEY"))
        logger.success("[Finnhub] LIVE client initialized")
    except ImportError:
        logger.warning("[Finnhub] finnhub-python not installed — using mock")

@tool
def portfolio_analysis(query: str) -> str:
    """Fetch financial metrics using the Finnhub API.
    Ref: Section 14.1.1, p.396"""
    symbol = query.split()[-1].upper()

    if finnhub_client is not None:
        try:
            financials = finnhub_client.company_basic_financials(symbol, "all")
            metrics = financials.get("metric", {})
            logger.success(f"[Finnhub] LIVE financials for {symbol}")
        except Exception:
            financials = MOCK_FINNHUB_FINANCIALS.get(symbol, MOCK_FINNHUB_FINANCIALS["AAPL"])
            metrics = financials.get("metric", {})
            logger.info(f"[Finnhub] Fallback to mock for {symbol}")
    else:
        financials = MOCK_FINNHUB_FINANCIALS.get(symbol, MOCK_FINNHUB_FINANCIALS["AAPL"])
        metrics = financials.get("metric", {})
        logger.info(f"[Finnhub] SIMULATED financials for {symbol}")

    return (
        f"Portfolio Analysis for {symbol}: "
        f"P/E Ratio: {metrics.get('peRatio')}, "
        f"Revenue Growth: {metrics.get('revenueGrowth')}, "
        f"52W High: {metrics.get('52WeekHigh')}, "
        f"52W Low: {metrics.get('52WeekLow')}"
    )

analysis_agent = make_react_agent(
    llm, [portfolio_analysis],
    "You are the Financial Analysis Agent. "
    "Perform portfolio analysis and compute financial metrics."
)

print(portfolio_analysis.invoke("Analyze AAPL"))
logger.success("Financial Analysis Agent initialized")

[18:22:44] [Chapter14] INFO [Simulation Mode] MockChatOpenAI initialized (model=gpt-4o-mini-2024-07-18)
[18:22:44] [Chapter14] INFO [Finnhub] SIMULATED financials for AAPL
Portfolio Analysis for AAPL: P/E Ratio: 28.5, Revenue Growth: 7.8, 52W High: 199.62, 52W Low: 143.9
[18:22:44] [Chapter14] SUCCESS Financial Analysis Agent initialized


## Cell 4: Financial News Agent

**Ref:** Section 14.1.1 (p.397)

The Financial News Agent provides qualitative context using Tavily search-based retrieval. In Simulation Mode, it returns chapter-derived mock news results.

In [6]:
# Cell 4: Financial News Agent
# Ref: Section 14.1.1, p.397
# Author: Imran Ahmad

@tool
def search_financial_news(query: str) -> str:
    """Search for financial news using Tavily or mock data.
    Ref: Section 14.1.1, p.397"""
    if config.is_live("TAVILY_API_KEY"):
        try:
            from langchain_community.tools.tavily_search import TavilySearchResults
            tavily_tool = TavilySearchResults(max_results=5)
            results = tavily_tool.invoke(query)
            logger.success(f"[Tavily] LIVE search: {len(results)} results")
            return json.dumps(results, indent=2)
        except Exception as e:
            logger.warning(f"[Tavily] API error: {e} — using mock")

    logger.info("[Tavily] SIMULATED news search")
    return json.dumps(MOCK_TAVILY_NEWS, indent=2)

financial_news_agent = make_react_agent(
    llm, [search_financial_news],
    "You are the Financial News Agent. "
    "Retrieve and summarize the latest financial news "
    "relevant to the user's query."
)

# Demo
news_result = json.loads(search_financial_news.invoke("technology sector outlook"))
for item in news_result[:3]:
    print(f"  * {item['title']} (score: {item['score']})")
logger.success("Financial News Agent initialized")

[18:22:44] [Chapter14] INFO [Simulation Mode] MockChatOpenAI initialized (model=gpt-4o-mini-2024-07-18)
[18:22:44] [Chapter14] INFO [Tavily] SIMULATED news search
  * Federal Reserve Signals Cautious Approach to Rate Adjustments (score: 0.95)
  * Technology Sector Posts Strong Q4 Earnings (score: 0.91)
  * Global Trade Outlook Improves Amid Diplomatic Progress (score: 0.87)
[18:22:44] [Chapter14] SUCCESS Financial News Agent initialized


## Cell 5: StateGraph Assembly and Streaming Execution

**Ref:** Section 14.1 (p.397–399)

The supervisor orchestrates the specialist agents through a LangGraph `StateGraph`. This loop-until-complete pattern ensures complex multi-source queries are fully resolved before generating a response.

**Topology:** `START` → `supervisor` → conditional edge to specialist or `FINISH` → `END`. Each specialist returns to the supervisor for re-evaluation.

In [7]:
# Cell 5: StateGraph Assembly and Streaming Execution
# Ref: Section 14.1, p.397-399
# Author: Imran Ahmad

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    next: str

def agent_node(state, agent, name):
    """Execute a specialist agent and wrap its result for the state graph.
    Ref: Section 14.1, p.397"""
    result = agent.invoke(state)
    return {"messages": [HumanMessage(
        content=result["messages"][-1].content, name=name
    )]}

market_data_node = partial(agent_node, agent=market_agent, name="Market_Data_Agent")
analysis_node = partial(agent_node, agent=analysis_agent, name="Financial_Analysis_Agent")
news_node = partial(agent_node, agent=financial_news_agent, name="News_Agent")

workflow = StateGraph(AgentState)
workflow.add_node("Market_Data_Agent", market_data_node)
workflow.add_node("Financial_Analysis_Agent", analysis_node)
workflow.add_node("News_Agent", news_node)
workflow.add_node("supervisor", supervisor_agent)

for member in members:
    workflow.add_edge(member, "supervisor")

conditional_map = {m: m for m in members}
conditional_map["FINISH"] = END

workflow.add_conditional_edges(
    "supervisor", lambda x: x["next"], conditional_map
)
workflow.add_edge(START, "supervisor")

graph = workflow.compile()
logger.success("Financial Advisory StateGraph compiled")

# Execute streaming query — Ref: p.398-399
MockStructuredChain.reset()
logger.info("Executing: 'Analyze the portfolio for AAPL.'")
inputs = {"messages": [HumanMessage(content="Analyze the portfolio for AAPL.")]}

for output in graph.stream(inputs, stream_mode="values"):
    if "messages" in output:
        last_msg = output["messages"][-1]
        sender = getattr(last_msg, "name", "system")
        content = last_msg.content[:120] if last_msg.content else "(routing)"
        logger.info(f"[{sender}] {content}")

logger.success("StateGraph execution complete")

[18:22:44] [Chapter14] SUCCESS Financial Advisory StateGraph compiled
[18:22:44] [Chapter14] INFO Executing: 'Analyze the portfolio for AAPL.'
[18:22:44] [Chapter14] INFO [None] Analyze the portfolio for AAPL.
[18:22:44] [Chapter14] INFO [Supervisor] Routing to: Market_Data_Agent
[18:22:44] [Chapter14] INFO [None] Analyze the portfolio for AAPL.
[18:22:44] [Chapter14] INFO [Market Data] SIMULATED data for ANALYZE THE PORTFOLIO FOR AAPL.
[18:22:44] [Chapter14] INFO [Market_Data_Agent] Market Data for AAPL: Price: $178.72, Market Cap: $2800000000000, P/E Ratio: 28.5, Day Range: $176.50-$179.80, Volume: 5
[18:22:44] [Chapter14] INFO [Supervisor] Routing to: Financial_Analysis_Agent
[18:22:44] [Chapter14] INFO [Market_Data_Agent] Market Data for AAPL: Price: $178.72, Market Cap: $2800000000000, P/E Ratio: 28.5, Day Range: $176.50-$179.80, Volume: 5
[18:22:44] [Chapter14] INFO [Finnhub] SIMULATED financials for RANGE.
[18:22:44] [Chapter14] INFO [Financial_Analysis_Agent] Portfolio Analysis

## Cell 6: Risk Assessment Framework

**Ref:** Section 14.1.2 (pp. 399–404)

Three levels of risk evaluation:
1. **Basic volatility classification** — `|dp| > 5` → HIGH, `|dp| > 2` → MODERATE, else LOW
2. **Composite risk scoring** (`RiskScorer`) — 0.4 × volatility + 0.35 × drawdown + 0.25 × VaR (0–10 scale)
3. **Client tolerance adjustment** (`assess_risk`) — Maps market risk against conservative/moderate/aggressive tolerance

---

> 📌 **Info Box — Forty-five minutes, four hundred forty million dollars (p. 399)**
>
> On August 1, 2012, Knight Capital Group deployed a software update to its automated trading system. A configuration error reactivated dormant code that began executing millions of unintended trades across 154 stocks. In forty-five minutes, the firm accumulated $7 billion in erroneous positions and lost approximately $440 million — nearly its entire market capitalization. Knight was rescued through an emergency capital raise but never fully recovered, merging with Getco LLC the following year. The incident remains the canonical warning for automated financial systems: **speed without safeguards is not an advantage — it is a liability.** Every compliance gate, risk threshold, and human checkpoint described in this chapter exists to prevent precisely this kind of cascading failure.

---

> 📌 **Info Box — Key risk metrics (p. 400)**
>
> **Value at Risk (VaR)** estimates the maximum expected loss over a given time horizon at a specified confidence level (e.g., a 1-day 95% VaR of $10,000 means there is a 5% chance of losing more than $10,000 in a single day). **Conditional Value at Risk (CVaR)**, also called expected shortfall, measures the average loss in the worst-case scenarios beyond the VaR threshold, making it more sensitive to tail risk. **Annualized volatility** expresses the standard deviation of returns scaled to a one-year period, providing a comparable measure of price instability across assets. **Maximum drawdown** captures the largest peak-to-trough decline over a period, reflecting the worst loss an investor would have experienced had they bought at the peak and sold at the trough.

---

> 📌 **Info Box — Production data feeds (p. 398)**
>
> The yfinance library is suitable for prototyping and demonstration but does not provide the reliability guarantees required in regulated financial environments. Production systems should source market data from commercial providers such as Bloomberg, Refinitiv (LSEG Data & Analytics), or FactSet, which offer contractual SLA commitments, real-time feeds with sub-second latency, and data quality controls. When evaluating a provider, confirm coverage for all required asset classes, jurisdictional data permissions, and API rate limits that will hold under production load.

In [8]:
# Cell 6: Risk Assessment Framework
# Ref: Section 14.1.2, p.399-404
# Author: Imran Ahmad

# ── 1. Basic Volatility Classification ──
# Ref: p.400 — thresholds: abs(dp) > 5 → HIGH, > 2 → MODERATE, else LOW

def risk_assessment(query: str) -> str:
    """Evaluate investment risk using real-time volatility metrics.
    Ref: Section 14.1.2, p.400"""
    symbol = query.split()[-1].upper()

    if finnhub_client is not None:
        try:
            quote = finnhub_client.quote(symbol)
            logger.success(f"[Risk] LIVE quote for {symbol}")
        except Exception:
            quote = MOCK_FINNHUB_QUOTES.get(symbol, MOCK_FINNHUB_QUOTES["AAPL"])
            logger.info(f"[Risk] Fallback to mock quote for {symbol}")
    else:
        quote = MOCK_FINNHUB_QUOTES.get(symbol, MOCK_FINNHUB_QUOTES["AAPL"])
        logger.info(f"[Risk] SIMULATED quote for {symbol}")

    price_change = quote.get("dp", 0)

    if abs(price_change) > 5:
        risk_level = "High Risk"
    elif abs(price_change) > 2:
        risk_level = "Moderate Risk"
    else:
        risk_level = "Low Risk"

    return (
        f"Risk Assessment for {symbol}: "
        f"Price Change: {price_change}%, "
        f"Risk Level: {risk_level}"
    )

for sym in ["AAPL", "GOOGL", "MSFT"]:
    print(risk_assessment(f"Assess {sym}"))

print()

# ── 2. Composite Risk Scoring ──
# Ref: p.400-401 — weights: 0.4 vol + 0.35 dd + 0.25 var

class RiskScorer:
    """Multi-dimensional risk scoring for portfolio positions.
    Ref: Section 14.1.2, p.400-401"""

    def compute_risk_score(self, symbol: str,
                           lookback_days: int = 90) -> dict:
        """Compute composite risk score incorporating
        volatility, drawdown, and VaR metrics."""
        try:
            if config.is_live("OPENAI_API_KEY"):
                import yfinance as yf
                ticker = yf.Ticker(symbol)
                hist = ticker.history(period=f"{lookback_days}d")
                if hist.empty:
                    raise ValueError("Empty history")
                logger.success(f"[RiskScorer] LIVE history for {symbol}")
            else:
                raise ValueError("Simulation mode")
        except Exception:
            mock_hist = generate_mock_price_history(symbol, days=lookback_days)
            hist = pd.DataFrame(mock_hist)
            logger.info(f"[RiskScorer] SIMULATED history for {symbol}")

        returns = hist["Close"].pct_change().dropna()

        # Ref: p.401
        volatility = returns.std() * np.sqrt(252)
        cumulative = (1 + returns).cumprod()
        rolling_max = cumulative.cummax()
        drawdown = (cumulative - rolling_max) / rolling_max
        max_drawdown = drawdown.min()
        var_95 = np.percentile(returns, 5)

        vol_score = min(volatility / 0.05, 10)
        dd_score = min(abs(max_drawdown) / 0.05, 10)
        var_score = min(abs(var_95) / 0.03, 10)

        composite = 0.4 * vol_score + 0.35 * dd_score + 0.25 * var_score

        return {
            "symbol": symbol,
            "annualized_volatility": round(float(volatility), 4),
            "max_drawdown": round(float(max_drawdown), 4),
            "var_95": round(float(var_95), 4),
            "composite_risk_score": round(float(composite), 2),
            "risk_category": self._categorize(float(composite)),
        }

    @staticmethod
    def _categorize(score: float) -> str:
        """Ref: p.401 — >= 7.0 HIGH, >= 4.0 MODERATE, else LOW"""
        if score >= 7.0:
            return "HIGH"
        elif score >= 4.0:
            return "MODERATE"
        return "LOW"

scorer = RiskScorer()
risk_result = scorer.compute_risk_score("AAPL")
print("--- Composite Risk Score ---")
for key, value in risk_result.items():
    print(f"  {key}: {value}")
print()

# ── 3. Client Tolerance Adjustment ──
# Ref: p.401-402

def assess_risk(stock_symbol: str, composite_score: float,
                client_risk_tolerance: str) -> dict:
    """Evaluate risk level adjusted for client tolerance.
    Ref: Section 14.1.2, p.401-402"""
    if composite_score >= 7.0:
        market_risk = "HIGH"
    elif composite_score >= 4.0:
        market_risk = "MODERATE"
    else:
        market_risk = "LOW"

    tolerance_map = {
        "conservative": {"HIGH": "UNACCEPTABLE", "MODERATE": "HIGH", "LOW": "MODERATE"},
        "moderate": {"HIGH": "HIGH", "MODERATE": "MODERATE", "LOW": "LOW"},
        "aggressive": {"HIGH": "MODERATE", "MODERATE": "LOW", "LOW": "LOW"},
    }

    adjusted_risk = tolerance_map.get(
        client_risk_tolerance, {}
    ).get(market_risk, market_risk)

    return {
        "symbol": stock_symbol,
        "market_risk": market_risk,
        "client_risk_tolerance": client_risk_tolerance,
        "adjusted_risk": adjusted_risk,
    }

print("--- Client Tolerance Adjustment ---")
for tolerance in ["conservative", "moderate", "aggressive"]:
    result = assess_risk("AAPL", risk_result["composite_risk_score"], tolerance)
    print(f"  {tolerance}: market={result['market_risk']} -> adjusted={result['adjusted_risk']}")

logger.success("Risk Assessment Framework complete")

[18:22:44] [Chapter14] INFO [Risk] SIMULATED quote for AAPL
Risk Assessment for AAPL: Price Change: 0.71%, Risk Level: Low Risk
[18:22:44] [Chapter14] INFO [Risk] SIMULATED quote for GOOGL
Risk Assessment for GOOGL: Price Change: 2.99%, Risk Level: Moderate Risk
[18:22:44] [Chapter14] INFO [Risk] SIMULATED quote for MSFT
Risk Assessment for MSFT: Price Change: -5.23%, Risk Level: High Risk

[18:22:44] [Chapter14] INFO [RiskScorer] SIMULATED history for AAPL
--- Composite Risk Score ---
  symbol: AAPL
  annualized_volatility: 0.209
  max_drawdown: -0.0899
  var_95: -0.0214
  composite_risk_score: 2.48
  risk_category: LOW

--- Client Tolerance Adjustment ---
  conservative: market=LOW -> adjusted=MODERATE
  moderate: market=LOW -> adjusted=LOW
  aggressive: market=LOW -> adjusted=LOW
[18:22:44] [Chapter14] SUCCESS Risk Assessment Framework complete


## Cell 7: Personalized Financial Planning and Compliance Gate

**Ref:** Section 14.1.3 (pp. 403–408)

The compliance gate implements **compliance-by-architecture**: it is structurally impossible for a non-compliant recommendation to reach the client. The `validate_compliance` node checks suitability and concentration limits (max 25%). If it fails, the `revise` node adjusts and loops back for re-validation.

**Table 14.1 (p. 403): Performance impact of AI-powered personalization in retail financial advisory**

| Metric | Before AI | After AI |
|:-------|:----------|:---------|
| Response time | 4-hour average | 30 seconds |
| Compliance accuracy | 95% | 99.99% |
| Client capacity | 100 per advisor | 1,000+ per advisor |
| Monthly revenue | $8,000 | $25,000 |

> 📌 **Industry Examples (p. 408):** JPMorgan Chase has invested in AI-driven summarization of market research, natural language understanding for client inquiries, and compliance monitoring that flags risky advisory patterns. Virgin Money's Redi agent demonstrates how agentic systems can elevate retail banking beyond static FAQs, handling nuanced financial questions involving transactional histories and account-linked decisions with context-aware personalization.

In [9]:
# Cell 7: Personalized Financial Planning and Compliance Gate
# Ref: Section 14.1.3, p.403-408
# Author: Imran Ahmad

class ClientProfileAgent:
    """Retrieves and contextualizes client financial profiles.
    Ref: Section 14.1.3, p.403-404"""

    def __init__(self, profile_store: dict):
        self.profiles = profile_store

    def get_contextualized_profile(self, client_id: str,
                                   query_context: str = "") -> dict:
        profile = self.profiles.get(client_id, {})
        if not profile:
            logger.warning(f"[ClientProfile] No profile found for {client_id}")
            return {}
        logger.info(f"[ClientProfile] Retrieved profile for {profile.get('name', client_id)}")
        return {
            "profile": profile,
            "risk_tolerance": profile.get("risk_tolerance"),
            "max_risk_tolerance": profile.get("max_risk_tolerance", 5.0),
            "investment_horizon": profile.get("investment_horizon"),
            "regulatory_constraints": profile.get("constraints", []),
        }

client_agent = ClientProfileAgent(MOCK_CLIENT_PROFILES)
profile_data = client_agent.get_contextualized_profile("retail_client_4521")
print(f"Client: {profile_data['profile']['name']}")
print(f"  Risk tolerance: {profile_data['risk_tolerance']}, Horizon: {profile_data['investment_horizon']}")
print()

# ── Compliance-Gated Advisory Workflow ──
# Ref: p.405-406

class AdvisoryState(TypedDict):
    messages: list
    client_profile: dict
    recommendation: dict
    compliance_result: dict
    final_response: str

policy_rules = {"max_concentration": 0.25}

def generate_recommendation(state: AdvisoryState):
    """Generate allocation based on client profile. Ref: p.405, p.406-407"""
    tolerance = state["client_profile"].get("risk_tolerance", "moderate")
    allocations = {
        "conservative": ({"us_equities": 0.25, "international_equities": 0.10,
                          "fixed_income": 0.50, "alternatives": 0.15}, 3.2),
        "aggressive": ({"us_equities": 0.55, "international_equities": 0.25,
                        "fixed_income": 0.10, "alternatives": 0.10}, 7.8),
        "moderate": ({"us_equities": 0.45, "international_equities": 0.20,
                      "fixed_income": 0.25, "alternatives": 0.10}, 6.2),
    }
    alloc, risk = allocations.get(tolerance, allocations["moderate"])
    logger.info(f"[Recommend] Generated allocation for {tolerance} client")
    return {"recommendation": {"allocation": alloc, "risk_score": risk,
                               "expected_annual_return": 0.078,
                               "max_drawdown_estimate": -0.18}}

def validate_compliance(state: AdvisoryState):
    """Validate against regulatory requirements. Ref: p.405-406"""
    rec = state["recommendation"]
    profile = state["client_profile"]
    issues = []
    max_tol = profile.get("max_risk_tolerance", 5.0)
    if rec["risk_score"] > max_tol:
        issues.append(f"SUITABILITY: Risk ({rec['risk_score']}) exceeds tolerance ({max_tol})")
    max_conc = policy_rules.get("max_concentration", 0.25)
    for asset, weight in rec["allocation"].items():
        if weight > max_conc:
            issues.append(f"CONCENTRATION: {asset} at {weight:.0%} exceeds {max_conc:.0%}")
    if issues:
        for i in issues:
            logger.warning(f"[Compliance] {i}")
    else:
        logger.success("[Compliance] All checks passed")
    return {"compliance_result": {"passed": len(issues) == 0, "issues": issues}}

def route_after_compliance(state: AdvisoryState):
    return "deliver" if state["compliance_result"]["passed"] else "revise"

def revise_recommendation(state: AdvisoryState):
    """Revise non-compliant recommendation. Ref: p.408"""
    rec = state["recommendation"]
    logger.info(f"[Revise] Adjusting recommendation")
    allocation = dict(rec["allocation"])
    max_conc = policy_rules.get("max_concentration", 0.25)
    # Cap all over-limit positions and redistribute evenly
    total_excess = 0.0
    for asset in list(allocation):
        if allocation[asset] > max_conc:
            total_excess += allocation[asset] - max_conc
            allocation[asset] = max_conc
    # Spread excess proportionally to assets still under limit
    if total_excess > 0:
        under = [a for a in allocation if allocation[a] < max_conc]
        if under:
            share = total_excess / len(under)
            for a in under:
                allocation[a] = round(min(allocation[a] + share, max_conc), 4)
    new_risk = min(rec["risk_score"],
                   state["client_profile"].get("max_risk_tolerance", 5.0))
    return {"recommendation": {**rec, "allocation": allocation, "risk_score": new_risk}}

def deliver_to_client(state: AdvisoryState):
    """Deliver validated recommendation. Ref: p.407"""
    rec = state["recommendation"]
    name = state["client_profile"].get("name", "Client")
    response = (f"Advisory Recommendation for {name}:\n"
                f"  Allocation: {json.dumps(rec['allocation'], indent=4)}\n"
                f"  Risk Score: {rec['risk_score']}\n"
                f"  Compliance: VALIDATED")
    logger.success(f"[Deliver] Recommendation delivered to {name}")
    print("\n" + response)
    return {"final_response": response}

compliance_workflow = StateGraph(AdvisoryState)
compliance_workflow.add_node("recommend", generate_recommendation)
compliance_workflow.add_node("comply", validate_compliance)
compliance_workflow.add_node("deliver", deliver_to_client)
compliance_workflow.add_node("revise", revise_recommendation)
compliance_workflow.add_edge(START, "recommend")
compliance_workflow.add_edge("recommend", "comply")
compliance_workflow.add_conditional_edges(
    "comply", route_after_compliance,
    {"deliver": "deliver", "revise": "revise"}
)
compliance_workflow.add_edge("revise", "comply")
compliance_workflow.add_edge("deliver", END)
compliance_graph = compliance_workflow.compile()
logger.success("Compliance-gated advisory workflow compiled")

# Execute for moderate client
MockStructuredChain.reset()
compliance_graph.invoke({
    "messages": [], "client_profile": MOCK_CLIENT_PROFILES["retail_client_4521"],
    "recommendation": {}, "compliance_result": {}, "final_response": "",
})

[18:22:44] [Chapter14] INFO [ClientProfile] Retrieved profile for Sarah Chen
Client: Sarah Chen
  Risk tolerance: moderate, Horizon: 10 years

[18:22:44] [Chapter14] SUCCESS Compliance-gated advisory workflow compiled
[18:22:44] [Chapter14] INFO [Recommend] Generated allocation for moderate client
[18:22:44] [Chapter14] WARNING [Compliance] CONCENTRATION: us_equities at 45% exceeds 25%
[18:22:44] [Chapter14] INFO [Revise] Adjusting recommendation
[18:22:44] [Chapter14] SUCCESS [Compliance] All checks passed
[18:22:44] [Chapter14] SUCCESS [Deliver] Recommendation delivered to Sarah Chen

Advisory Recommendation for Sarah Chen:
  Allocation: {
    "us_equities": 0.25,
    "international_equities": 0.25,
    "fixed_income": 0.25,
    "alternatives": 0.2
}
  Risk Score: 6.2
  Compliance: VALIDATED


{'messages': [],
 'client_profile': {'client_id': 'retail_client_4521',
  'name': 'Sarah Chen',
  'risk_tolerance': 'moderate',
  'max_risk_tolerance': 6.5,
  'investment_horizon': '10 years',
  'initial_investment': 50000,
  'financial_goals': ['moderate growth', 'retirement savings'],
  'constraints': ['No tobacco stocks', 'ESG preference'],
  'age': 35,
  'income_bracket': 'middle',
  'experience_level': 'intermediate'},
 'recommendation': {'allocation': {'us_equities': 0.25,
   'international_equities': 0.25,
   'fixed_income': 0.25,
   'alternatives': 0.2},
  'risk_score': 6.2,
  'expected_annual_return': 0.078,
  'max_drawdown_estimate': -0.18},
 'compliance_result': {'passed': True, 'issues': []},
 'final_response': 'Advisory Recommendation for Sarah Chen:\n  Allocation: {\n    "us_equities": 0.25,\n    "international_equities": 0.25,\n    "fixed_income": 0.25,\n    "alternatives": 0.2\n}\n  Risk Score: 6.2\n  Compliance: VALIDATED'}

## Cell 8: RetailAdvisor Case Study

**Ref:** Section 14.1.4 (p.406–410)

End-to-end demonstration: *"I have $50,000 to invest and want moderate growth over the next ten years."* Shows the inter-agent JSON communication protocol (p.407), risk scoring, and compliance validation.

In [10]:
# Cell 8: RetailAdvisor Case Study
# Ref: Section 14.1.4, p.406-410
# Author: Imran Ahmad

logger.info("=" * 60)
logger.info("CASE STUDY: RetailAdvisor")
logger.info("=" * 60)

# Step 1: Client query from chapter p.408
query = "I have $50,000 to invest and want moderate growth over the next ten years."
logger.info(f"[Client Query] {query}")

# Step 2: Client profile
cp = client_agent.get_contextualized_profile("retail_client_4521")
print(f"\nClient: {cp['profile']['name']}")
print(f"  Investment: ${cp['profile']['initial_investment']:,}, Horizon: {cp['investment_horizon']}")

# Step 3: Risk scoring
print("\n--- Portfolio Risk Assessment ---")
for sym in ["AAPL", "MSFT", "GOOGL"]:
    s = scorer.compute_risk_score(sym)
    print(f"  {sym}: composite={s['composite_risk_score']}, category={s['risk_category']}")

# Step 4: Inter-agent message protocol (p.407)
print("\n--- Inter-Agent Communication Protocol (p.407) ---")
print(json.dumps(MOCK_INTER_AGENT_MESSAGE, indent=2))

# Step 5: Compliance-gated pipeline
MockStructuredChain.reset()
print("\n--- Compliance-Gated Advisory Pipeline ---")
result = compliance_graph.invoke({
    "messages": [], "client_profile": MOCK_CLIENT_PROFILES["retail_client_4521"],
    "recommendation": {}, "compliance_result": {}, "final_response": "",
})

# Step 6: Test with conservative client (may trigger revisions)
print("\n--- Conservative client (may trigger compliance revision) ---")
compliance_graph.invoke({
    "messages": [], "client_profile": MOCK_CLIENT_PROFILES["retail_client_7832"],
    "recommendation": {}, "compliance_result": {}, "final_response": "",
})

logger.success("RetailAdvisor Case Study complete")

[18:22:44] [Chapter14] INFO ============================================================
[18:22:44] [Chapter14] INFO CASE STUDY: RetailAdvisor
[18:22:44] [Chapter14] INFO ============================================================
[18:22:44] [Chapter14] INFO [Client Query] I have $50,000 to invest and want moderate growth over the next ten years.
[18:22:44] [Chapter14] INFO [ClientProfile] Retrieved profile for Sarah Chen

Client: Sarah Chen
  Investment: $50,000, Horizon: 10 years

--- Portfolio Risk Assessment ---
[18:22:44] [Chapter14] INFO [RiskScorer] SIMULATED history for AAPL
  AAPL: composite=2.48, category=LOW
[18:22:44] [Chapter14] INFO [RiskScorer] SIMULATED history for MSFT
  MSFT: composite=2.48, category=LOW
[18:22:44] [Chapter14] INFO [RiskScorer] SIMULATED history for GOOGL
  GOOGL: composite=2.48, category=LOW

--- Inter-Agent Communication Protocol (p.407) ---
{
  "sender_id": "portfolio_construction_agent",
  "recipient_id": "compliance_agent",
  "message_type": "re

## Summary

In this workshop agent you built and executed the Financial Advisory Agent end-to-end:

- **Supervisor pattern** — a policy-aware orchestrator routes queries to Market Data, Financial Analysis, and News specialists through a LangGraph `StateGraph` (Figure 14.1)
- **Composite risk scoring** — 0.4 × volatility + 0.35 × drawdown + 0.25 × VaR on a 0–10 scale, adjusted for the client's stated tolerance
- **Compliance-by-architecture** — the validation gate makes it structurally impossible for a non-compliant recommendation to reach the client; the revise node loops until all checks pass
- **RetailAdvisor case study** — a $50,000 moderate-growth mandate carried from client profile to a validated allocation, including the inter-agent JSON communication protocol (p. 407)

**Book reference:** *30 Agents Every AI Engineer Must Build* — Imran Ahmad (Packt Publishing, 2026), Chapter 14, Section 14.1 (pp. 392–408). Section 14.2 applies the same compliance-first thinking to the Legal Intelligence Agent.
